[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/05_attention.ipynb)

# 🔴 Hard: Softmax Attention

Implement the core attention mechanism used in Transformers.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Signature
```python
def scaled_dot_product_attention(
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
) -> torch.Tensor:   # (batch, seq_q, d_v)
    ...
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- You **may** use `torch.softmax` and `torch.bmm`
- Must support autograd
- Must handle cross-attention (seq_q ≠ seq_k)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.3 MB/s eta 0:00:00


In [2]:
import torch
import math

In [21]:
import torch
import math

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scaling_factor = torch.tensor(math.sqrt(d_k), dtype=Q.dtype, device=Q.device)

    attn_mat = torch.bmm(Q, K.transpose(-2, -1)) / scaling_factor
    print(attn_mat.shape)
    attn_score = torch.softmax(attn_mat, dim=-1)
    print(attn_score.shape)

    proj = torch.bmm(attn_score, V)
    print(proj.shape)
    return proj

In [22]:
# 🧪 Debug
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)          # should be (2, 4, 8)
print("Has NaN?    ", torch.isnan(out).any().item())  # should be False
print("Has Inf?    ", torch.isinf(out).any().item())  # should be False

# Cross-attention: seq_q != seq_k
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attn shape:", out2.shape)     # should be (1, 3, 32)

torch.Size([2, 4, 4])
torch.Size([2, 4, 4])
torch.Size([2, 4, 8])
Output shape: torch.Size([2, 4, 8])
Has NaN?     False
Has Inf?     False
torch.Size([1, 3, 5])
torch.Size([1, 3, 5])
torch.Size([1, 3, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [23]:
# ✅ SUBMIT
from torch_judge import check, hint
check("attention")


🧪 Testing: Softmax Attention (Hard)
──────────────────────────────────────────────────
torch.Size([2, 4, 4])
torch.Size([2, 4, 4])
torch.Size([2, 4, 8])
  ✅ [1/4] Output shape (3.6ms)
torch.Size([2, 4, 4])
torch.Size([2, 4, 4])
torch.Size([2, 4, 8])
  ✅ [2/4] Numerical correctness (2.6ms)
torch.Size([2, 4, 4])
torch.Size([2, 4, 4])
torch.Size([2, 4, 8])
  ✅ [3/4] Gradient check (1.6ms)
torch.Size([1, 3, 5])
torch.Size([1, 3, 5])
torch.Size([1, 3, 32])
  ✅ [4/4] Cross-attention (seq_q != seq_k) (0.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (8.0ms total)
  Progress saved. Run status() to see your dashboard.



In [15]:
hint("attention")


💡 Hint for Softmax Attention:
   scores = Q @ K^T / sqrt(d_k), then softmax(scores, dim=-1) @ V. Use torch.bmm for batched matmul.



1. Difference between torch.bmm vs torch.matmul vs @
2. K.shape[0] vs K.shape[-1]
3. Note there was an error in mumerical correctness becuase you did not use torch.bmm very very important